# Μάθημα 08 - Σχεδιαστικό Πρότυπο Πολυ-Πρακτόρων


## Εγκατάσταση


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Γιατί Πολλαπλά Συστήματα Πρακτόρων;

Οι εργασίες στον πραγματικό κόσμο, όπως ο προγραμματισμός ταξιδιού, απαιτούν πολλές διαφορετικές εξειδικεύσεις — logistics, τοπική γνώση, προϋπολογισμό, και άλλα. Ένας μόνος πράκτορας που προσπαθεί να χειριστεί τα πάντα γρήγορα γίνεται δύσχρηστος.

Τα συστήματα με πολλούς πράκτορες λύνουν αυτό το πρόβλημα μέσω της **εξειδίκευσης**: κάθε πράκτορας επικεντρώνεται σε έναν τομέα εξειδίκευσης, παράγοντας αποτελέσματα υψηλότερης ποιότητας από έναν γενικευμένο. Επίσης, βελτιώνουν την **κλιμακωσιμότητα** — μπορείτε να προσθέσετε νέους πράκτορες (π.χ. ειδικό σε πτήσεις, κριτικό εστιατορίων) χωρίς να ξαναγράψετε τη υπάρχουσα ροή εργασίας. Οι πράκτορες συνεργάζονται μέσα από μια δομημένη αλυσίδα, μεταφέροντας το πλαίσιο από τον έναν στον επόμενο.


## Δημιουργία Εξειδικευμένων Πρακτόρων


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Δημιουργία μιας Ακολουθιακής Ροής Εργασιών

Το `WorkflowBuilder` σας επιτρέπει να συνδέσετε πράκτορες σε έναν προσανατολισμένο γράφο. Εδώ δημιουργούμε έναν απλό αγωγό δύο βημάτων: ο **TravelPlanner** συντάσσει το δρομολόγιο, στη συνέχεια ο **TravelConcierge** το αναθεωρεί και το βελτιώνει.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Προσθήκη Περισσότερων Πρακτόρων στη Ροή Εργασίας

Ένα από τα μεγαλύτερα πλεονεκτήματα του μοτίβου πολλαπλών πρακτόρων είναι πόσο εύκολη είναι η επέκτασή του. Παρακάτω προσθέτουμε έναν πράκτορα **BudgetReviewer** που ελέγχει το σχέδιο σε σχέση με τον προϋπολογισμό του ταξιδιώτη, επισημαίνει αντικείμενα που μπορεί να αυξήσουν τα έξοδα πέρα από το όριο και προτείνει εναλλακτικές που εξοικονομούν χρήματα. Η ροή εργασίας τώρα εκτελεί τρεις πράκτορες κατά σειρά:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να:

1. **Δημιουργείτε εξειδικευμένους πράκτορες** — ο καθένας με έναν συγκεκριμένο ρόλο (σχεδιασμός, concierge, ανασκόπηση προϋπολογισμού).
2. **Διασυνδέετε τους πράκτορες σε μια διαδοχική ροή εργασίας** χρησιμοποιώντας το `WorkflowBuilder` και το `add_edge`.
3. **Μεταδίδετε ροή δεδομένων** από μια αλυσίδα πολλαπλών πρακτόρων, παρακολουθώντας ποιος πράκτορας μιλάει.
4. **Επεκτείνετε μια ροή εργασίας** προσθέτοντας νέους πράκτορες στην αλυσίδα, χωρίς να τροποποιείτε τους υπάρχοντες.

Το πρότυπο σχεδίασης πολλαπλών πρακτόρων διατηρεί κάθε πράκτορα απλό, ενώ παράγει πιο πλούσια, πιο διεξοδικά ελεγμένα αποτελέσματα από ό,τι θα μπορούσε να πετύχει οποιοσδήποτε μεμονωμένος πράκτορας μόνος του.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
